# Build Trace Analysis - Initial Exploration

This notebook demonstrates safe loading and initial analysis of Clang `-ftime-trace` JSON files.

**Key Features:**
- Streaming JSON parsing to avoid memory issues
- Analysis of a single file as a test case
- Identification of template instantiation patterns
- Basic statistics and visualizations

**Dataset:**
- 4,484 JSON trace files (~46 GB total)
- Located in `build-trace/` directory
- Chrome Trace Event Format

In [ ]:
# Standard library imports
import sys
from pathlib import Path
from collections import Counter

# Third-party imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add utils to path - handle both Jupyter and VSCode notebook execution
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    # Running from notebooks directory
    utils_path = notebook_dir.parent / "utils"
else:
    # Running from project root or other location
    utils_path = (
        Path(__file__).parent.parent / "utils"
        if "__file__" in globals()
        else notebook_dir / "script" / "build_analysis" / "utils"
    )

sys.path.insert(0, str(utils_path))
print(f"Utils path: {utils_path}")
print(f"Utils exists: {utils_path.exists()}")

# Import our utilities
from trace_parser import (
    iter_trace_files,
    stream_events,
    load_trace_metadata,
    get_template_events,
    aggregate_by_name,
    get_top_events,
    extract_template_detail,
    microseconds_to_milliseconds,
)

# Configure plotting
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("✓ Imports successful")

## 1. Dataset Overview

Let's start by examining the trace files available.

In [ ]:
# Path to trace directory (relative to notebook location)
TRACE_DIR = Path.cwd().parent.parent.parent / "build-trace"

print(f"Trace directory: {TRACE_DIR}")
print(f"Exists: {TRACE_DIR.exists()}")

# Count trace files
trace_files = list(iter_trace_files(TRACE_DIR))
print(f"\nTotal trace files: {len(trace_files):,}")

# Show first few files
print("\nFirst 5 trace files:")
for i, f in enumerate(trace_files[:5], 1):
    rel_path = f.relative_to(TRACE_DIR)
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"{i}. {rel_path} ({size_mb:.2f} MB)")

## 2. Single File Analysis

Let's analyze a single trace file in detail to understand the data structure.

In [ ]:
# Select a sample file for detailed analysis
sample_file = trace_files[0]
print(f"Analyzing: {sample_file.name}")
print(f"Size: {sample_file.stat().st_size / (1024 * 1024):.2f} MB")

# Load metadata (fast, doesn't load events)
metadata = load_trace_metadata(sample_file)
print(f"\nMetadata: {metadata}")

### 2.1 Event Type Distribution

In [ ]:
# Count events by type (streaming - memory safe)
event_names = Counter()
total_events = 0

for event in stream_events(sample_file):
    event_names[event.get("name", "Unknown")] += 1
    total_events += 1

print(f"Total events: {total_events:,}")
print(f"Unique event types: {len(event_names)}")

# Show top event types
print("\nTop 15 event types by frequency:")
for name, count in event_names.most_common(15):
    pct = (count / total_events) * 100
    print(f"  {name:40s} {count:6,} ({pct:5.2f}%)")

In [ ]:
# Visualize event type distribution
top_events_df = pd.DataFrame(
    event_names.most_common(20), columns=["Event Type", "Count"]
)

plt.figure(figsize=(14, 8))
sns.barplot(data=top_events_df, x="Count", y="Event Type", palette="viridis")
plt.title(f"Top 20 Event Types in {sample_file.name}", fontsize=14, fontweight="bold")
plt.xlabel("Count", fontsize=12)
plt.ylabel("Event Type", fontsize=12)
plt.tight_layout()
plt.show()

### 2.2 Template Instantiation Analysis

In [ ]:
# Analyze template-related events
template_event_names = [
    "InstantiateClass",
    "InstantiateFunction",
    "InstantiateVariable",
    "ParseTemplate",
]

template_counts = {name: event_names.get(name, 0) for name in template_event_names}
total_template_events = sum(template_counts.values())

print("Template-related events:")
for name, count in template_counts.items():
    pct = (count / total_events) * 100 if total_events > 0 else 0
    print(f"  {name:30s} {count:6,} ({pct:5.2f}%)")

print(f"\nTotal template events: {total_template_events:,}")
print(
    f"Template events as % of total: {(total_template_events / total_events) * 100:.2f}%"
)

### 2.3 Duration Analysis

In [ ]:
# Get top events by duration
print("Top 20 longest-running events:\n")

top_by_duration = get_top_events(stream_events(sample_file), n=20, sort_by="dur")

for i, event in enumerate(top_by_duration, 1):
    name = event.get("name", "Unknown")
    dur_ms = microseconds_to_milliseconds(event.get("dur", 0))
    detail = extract_template_detail(event) or ""

    # Truncate detail if too long
    if len(detail) > 60:
        detail = detail[:57] + "..."

    print(f"{i:2d}. {name:30s} {dur_ms:8.2f} ms  {detail}")

In [ ]:
# Aggregate statistics by event type
print("Aggregating event statistics (this may take a moment)...")

aggregated = aggregate_by_name(stream_events(sample_file))

# Convert to DataFrame for easier analysis
agg_df = pd.DataFrame.from_dict(aggregated, orient="index")
agg_df.index.name = "event_name"
agg_df.reset_index(inplace=True)

# Convert durations to milliseconds
for col in ["total_duration", "avg_duration", "max_duration", "min_duration"]:
    agg_df[f"{col}_ms"] = agg_df[col].apply(microseconds_to_milliseconds)

# Sort by total duration
agg_df_sorted = agg_df.sort_values("total_duration_ms", ascending=False)

print("\nTop 15 event types by total duration:")
print(
    agg_df_sorted[
        [
            "event_name",
            "count",
            "total_duration_ms",
            "avg_duration_ms",
            "max_duration_ms",
        ]
    ]
    .head(15)
    .to_string(index=False)
)

In [ ]:
# Visualize duration distribution for top event types
top_15 = agg_df_sorted.head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total duration
sns.barplot(
    data=top_15,
    x="total_duration_ms",
    y="event_name",
    hue="event_name",
    ax=axes[0],
    palette="rocket",
    legend=False,
)
axes[0].set_title("Total Duration by Event Type", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Total Duration (ms)", fontsize=11)
axes[0].set_ylabel("Event Type", fontsize=11)

# Average duration
sns.barplot(
    data=top_15,
    x="avg_duration_ms",
    y="event_name",
    hue="event_name",
    ax=axes[1],
    palette="mako",
    legend=False,
)
axes[1].set_title("Average Duration by Event Type", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Average Duration (ms)", fontsize=11)
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

### 2.4 Template Instantiation Deep Dive

In [ ]:
# Analyze template instantiations specifically
print("Analyzing template instantiations...\n")

template_details = Counter()
template_durations = []

for event in get_template_events(stream_events(sample_file)):
    detail = extract_template_detail(event)
    if detail:
        template_details[detail] += 1
        template_durations.append(
            {
                "template": detail,
                "duration_ms": microseconds_to_milliseconds(event.get("dur", 0)),
                "event_type": event.get("name", "Unknown"),
            }
        )

print(f"Unique templates instantiated: {len(template_details):,}")
print("\nTop 15 most frequently instantiated templates:")
for template, count in template_details.most_common(15):
    # Truncate long template names
    display_name = template if len(template) <= 80 else template[:77] + "..."
    print(f"  {count:4d}x  {display_name}")

In [ ]:
# Find slowest template instantiations
if template_durations:
    template_df = pd.DataFrame(template_durations)
    template_df_sorted = template_df.sort_values("duration_ms", ascending=False)

    print("Top 20 slowest template instantiations:\n")
    for i, row in enumerate(template_df_sorted.head(20).itertuples(), 1):
        template = (
            row.template if len(row.template) <= 80 else row.template[:77] + "..."
        )
        print(f"{i:2d}. {row.duration_ms:8.2f} ms  [{row.event_type}]  {template}")
else:
    print("No template instantiation data found.")

## 3. Summary Statistics

In [ ]:
# Calculate total compilation time
total_duration_ms = agg_df["total_duration_ms"].sum()
total_duration_sec = total_duration_ms / 1000

print("=" * 60)
print(f"SUMMARY: {sample_file.name}")
print("=" * 60)
print(f"Total events:              {total_events:,}")
print(f"Unique event types:        {len(event_names):,}")
print(
    f"Template events:           {total_template_events:,} ({(total_template_events / total_events) * 100:.1f}%)"
)
print(f"Unique templates:          {len(template_details):,}")
print(
    f"Total compilation time:    {total_duration_sec:.2f} seconds ({total_duration_sec / 60:.2f} minutes)"
)
print(f"File size:                 {sample_file.stat().st_size / (1024 * 1024):.2f} MB")
print("=" * 60)

## 4. Next Steps

This initial exploration demonstrates:
- ✓ Safe streaming of large JSON files
- ✓ Event type distribution analysis
- ✓ Template instantiation identification
- ✓ Duration-based performance analysis

**Recommended next analyses:**
1. **Multi-file aggregation**: Analyze patterns across all 4,484 trace files
2. **Template hierarchy**: Build dependency graphs of template instantiations
3. **Hotspot identification**: Find the most expensive templates across the entire build
4. **Comparative analysis**: Compare build times before/after code changes
5. **Visualization**: Create interactive timelines and flame graphs

See notebooks:
- `02_template_analysis.ipynb` - Deep dive into template metaprogramming costs
- `03_visualization.ipynb` - Interactive charts and timelines